# 传输介质

## 简介


**Scikit-rf** 支持传输线模型的 S 参数仿真，如空心波导、同轴电缆、微带线和自由空间等。

[Media](../api/media/index.rst) 对象存储介质的属性，如*传播常数*和*特性阻抗*。网络对象（如给定长度的传输线）从介质生成。

本教程说明如何使用几种介质创建网络。它还解释了有关网络*端口阻抗*和*特性阻抗*的最常见问题。

拉丁语单词 *media* 的单数是 *medium*，意思是中间或中心。为了清晰起见，在 scikit-rf 中，*media* 用于单数和复数。

##  RG58C/U 同轴电缆

所有介质构造函数都有两个共同参数：

* `frequency`（必需）
* `z0_port`   （可选）

频率轴 `frequency` 是一个定义介质频带的 `Frequency` 对象。

*端口阻抗* `z0_port` 是可选的，用于将介质*特性阻抗* `z0` 重新归一化到仿真测量的*端口阻抗*。

如果未指定*端口阻抗* `z0_port`，则传输线具有介质的原始*特性阻抗* `z0`。

以下是一个 RG58C/U 柔性同轴电缆介质的示例。

### 仿真 VNA 测量
![coaxial measurement](figures/media_coax_measurement.svg)

In [ ]:
# various initialization
%matplotlib inline
import matplotlib.pyplot as plt

import skrf as rf

# import the desired media and the frequency axis
from skrf import Frequency
from skrf.media import Coaxial

rf.stylely()


# frequency
f_rg58 = Frequency(1, 5, 101, 'GHz')

# media with z0_port the port impedance of the VNA
rg58 = Coaxial(f_rg58, Dint = 0.91e-3, Dout = 2.95e-3, epsilon_r = 2.3, z0_port = 50)
print(rg58)

该介质的*特性阻抗* `z0` 约为 47 欧姆。*端口阻抗* `z0_port` 为 50 欧姆。传播常数 `gamma` 也从介质参数计算得出。这些属性定义了传输线模型。

In [ ]:
print(f'z0 = {rg58.z0[0]}')
print(f'z0_port = {rg58.z0_port[0]}')
print(f'gamma = {rg58.gamma[0]}')

让我们创建一个对应于 100 毫米长度同轴电缆的传输线网络。

In [ ]:
rg58_line = rg58.line(100, unit = 'mm', name = '100 mm, z0 Ohm')
print(rg58_line)

介质的*特性阻抗* `z0` 已重新归一化到*端口阻抗* `z0_port`。网络具有等于 `z0_port` 的端口阻抗 `z0`。

在某些情况下，需要任意阻抗的传输线而不创建多个介质。可以在构造时覆盖传输线的阻抗。结果网络重新归一化到 `z0_port`。

In [ ]:
rg58_25ohm_line = rg58.line(100, unit = 'mm', z0 = 25, name = '100 mm, 25 Ohm')
print(rg58_25ohm_line)

让我们绘制两条传输线的 S 参数。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize = (8, 3.5))

# return loss
rg58_line.plot_s_db(0, 0, ax = axes[0])
rg58_25ohm_line.plot_s_db(0, 0, ax = axes[0])
axes[0].set_title('Return Loss')
rg58_line.plot_s_db(1, 0, ax = axes[1])
rg58_25ohm_line.plot_s_db(1, 0, ax = axes[1])
axes[1].set_title('Insertion Loss')
plt.tight_layout()

回波损耗和插入损耗显示了*端口阻抗* `z0_port` 与传输线*特性阻抗* `z0` 之间失配的影响，该阻抗可以由几何形状定义或强制为任意值。

## 基本示例

让我们创建两个网络，一段 50 欧姆平面微带线和一段 WR-10 矩形波导。

| 微带线 | 矩形波导 |
| :-: | :-: |
| ![microstripline](figures/media_mline.svg) | ![rectangular waveguide](figures/media_rectangularwaveguide.svg) |

首先，使用模型参数创建两个 [Media](../api/media/index.rst) 对象。

### 介质对象

| 微带线 | | | WR-10 矩形波导 | | |
| :- | :- | :- | :- | :- | :- |
| 走线宽度 | `w` | 3 mm | 孔径宽度 | `a` | 100 mil |
| 走线厚度 | `t` |  35 um| 孔径高度 | `b` | 50 mil |
| 基板高度 | `h` | 1.6 mm | - | - | - |
| 相对介电常数（FR-4） | `ep_r` | 4.5  | 相对介电常数（空气） | `ep_r`| 1.0 |
| 电阻率（铜） | `rho` | 1.68e-08 欧姆 * 米  |  电阻率（铜） | `rho` | 1.68e-08 欧姆 * 米 |

In [ ]:
# various initialization
%matplotlib inline
import matplotlib.pyplot as plt

import skrf as rf

# import the desired media and the frequency axis
from skrf import Frequency
from skrf.constants import to_meters
from skrf.media import MLine, RectangularWaveguide

rf.stylely()

# create frequency axes
f_mlin = Frequency(0.1, 10,1001, 'GHz')
f_wr10 = Frequency(75, 110, 1001,'GHz')

# create media from parameters
mlin  =  MLine(f_mlin, w = 3e-3, h = 1.6e-3, t = 35e-6, ep_r = 4.5,  rho = 1.68e-08)
print(mlin)
wr10  = RectangularWaveguide(f_wr10, a = to_meters(100, 'mil'), b = to_meters(50, 'mil'), ep_r = 1.0,  rho = 1.68e-08)
print(wr10)

### 传输线创建

其次，使用 [Media](../api/media/index.rst) 对象生成对应于两种介质各 100 毫米长度的网络。

In [ ]:
# create the transmission line networks
mlin_100 = mlin.line(100e-3, unit = 'm', name = 'mlin 100mm')
print(mlin_100)
wr10_100 = wr10.line(100e-3, unit = 'm', name = 'wr10 100mm')
print(wr10_100)

### 绘制结果

传输线的 S 参数绘制在下图中。

In [ ]:
# prepare figure
fig1, axes = plt.subplots(2, 2, figsize = (8, 6))

# plot miscrostipline
mlin_100.plot_s_mag(0, 0, ax = axes[0,0])
mlin_100.plot_s_db(1, 0, ax = axes[0,1])

# plot rectangular waveguide
wr10_100.plot_s_mag(0, 0, ax = axes[1,0])
wr10_100.plot_s_db(1, 0, ax = axes[1,1])

# resize plot nicely
axes[0, 0].set_ylim((-1, 1))
axes[1, 0].set_ylim((-1, 1))
fig1.tight_layout()

传输线的插入损耗 S21 是频率相关的，但 S11 幅度是恒定的且为零。S11 的绝对幅度以对数形式绘制，因为 log(0) 未定义。

*为什么回波损耗 S11 没有显示在矢量网络分析仪测量中观察到的形状？*

这是因为在介质构造时未指定*端口阻抗* `z0_port`。在这种情况下，*特性阻抗* `z0` 用作*端口阻抗*，网络在整个频带上与自身完全匹配。

*特性阻抗*通常是频率相关的参数。具有频率相关的*端口阻抗*在电磁仿真中经常遇到。实际测量使用恒定的*端口阻抗*。

![simulation](figures/media_simulation.svg)

传输线的*端口阻抗* `z0` 和介质的*特性阻抗* `z0` 绘制如下。这些参数是频率相关的。在这两种情况下，传输线的*端口阻抗*等于介质的*特性阻抗*。

`Network` 对象的*端口阻抗*也是 `z0`，但它不是*特性阻抗*。`Network` S 参数归一化到其*端口阻抗*。

In [ ]:
# prepare figure
fig2, axes = plt.subplots(1, 2, figsize = (8, 3.5))

# plot miscrostipline
axes[0].plot(mlin_100.frequency.f_scaled, mlin_100.z0[:, 0].real, marker = '.',
             label = f'line {mlin_100.name}  port z0')
axes[0].plot(mlin.frequency.f_scaled, mlin.z0.real,
             label = 'media mlin characteristic z0')
axes[0].set_ylabel('Impedance (Ohm)')
axes[0].set_xlabel(f'Frequency ({mlin.frequency.unit})')
axes[0].set_title('Microstripline')
axes[0].legend()

# plot rectangular waveguide
axes[1].plot(wr10_100.frequency.f_scaled, wr10_100.z0[:, 0].real, marker = '.',
             label = f'line {wr10_100.name} port z0')
axes[1].plot(wr10.frequency.f_scaled, wr10.z0.real,
             label = 'media wr10 characteristic z0')
axes[1].set_ylabel('Impedance (Ohm)')
axes[1].set_xlabel(f'Frequency ({wr10.frequency.unit})')
axes[1].set_title('WR-10')
axes[1].legend()

# resize plot nicely
fig2.tight_layout()

### 类似测量的微带线

微带线的 S 参数测量使用 VNA（矢量网络分析仪）完成。VNA 具有已知的*端口阻抗* - 通常为 50 欧姆 - 和同轴连接。

使用同轴到微带线过渡，VNA 在同轴电缆末端进行校准。在此示例中，让我们假设理想的过渡（无长度，两侧完美匹配）。

因此，过渡只是 VNA *端口阻抗* `z0_port` 和传输线*特性阻抗* `z0` 之间的失配。

![mline measurement](figures/media_mline_measurement.svg)

要获得具有 50 欧姆*端口阻抗*的 S 参数网络，可以在介质对象构造时指定*端口阻抗* `z0_port`，或者可以将传输线从*特性阻抗*重新归一化到*端口阻抗*。

In [ ]:
# renormalization method
mlin_100_measured1 = mlin_100.copy()
mlin_100_measured1.renormalize([50, 50])
mlin_100_measured1.name = f'{mlin_100.name} renormalize'
print(mlin_100_measured1)

# port impedance specified at media construction method
mlin_meas  =  MLine(f_mlin, w=3e-3, h=1.6e-3, t=35e-6, ep_r=4.5,  rho=1.68e-08, z0_port=50)
mlin_100_measured2 = mlin_meas.line(100e-3, unit = 'm', name = 'mlin 100mm z0_port')
print(mlin_100_measured2)

下图显示微带线的*特性阻抗*现在嵌入在具有 50 欧姆*端口阻抗*的网络中。这相当于使用理想同轴到微带线过渡的 VNA 测量。

In [ ]:
# prepare figure
fig3, axes = plt.subplots(2, 2, figsize = (8, 6))
gs = axes[1, 0].get_gridspec()
for ax in axes[1, :]:
    ax.remove()
axbig = fig3.add_subplot(gs[1, :])

# plot return loss
mlin_100_measured1.plot_s_db(0, 0, ax=axes[0,0])
mlin_100_measured2.plot_s_db(0, 0, ax=axes[0,0])

# plot insertion loss
mlin_100_measured1.plot_s_db(1, 0, ax=axes[0,1])
mlin_100_measured2.plot_s_db(1, 0, ax=axes[0,1])

# plot port and characteristic impedances
axbig.plot(mlin_100_measured1.frequency.f_scaled, mlin_100_measured1.z0[:, 0].real,
           marker = 'd', markevery = 30, label = f'line {mlin_100_measured1.name} z0')
axbig.plot(mlin_100_measured2.frequency.f_scaled, mlin_100_measured2.z0[:, 0].real,
           marker = 'x', markevery = 30, label = f'line {mlin_100_measured2.name} z0')
axbig.plot(mlin.frequency.f_scaled, mlin.z0.real, label = 'media mlin z0')
axbig.set_ylabel('Impedance (Ohm)')
axbig.set_xlabel(f'Frequency ({mlin.frequency.unit})')
axbig.legend()

# resize plot nicely
fig3.tight_layout()

### 类似测量的 WR-10 波导

空心波导的 S 参数测量使用 VNA（矢量网络分析仪）完成。VNA 具有已知的端口阻抗 - 通常为 50 欧姆 - 和同轴连接。

使用同轴到波导过渡。过渡在波导界面处校准。因此，VNA *端口阻抗*覆盖了波导的*特性阻抗*。

测量将存储*端口阻抗*而不是*特性阻抗*，后者丢失。这不是归一化。未测量传输线的实际*特性阻抗*。此方法特定于空心波导。

![waveguide measurement](figures/media_waveguide_measurement.svg)

要获得具有 50 欧姆端口阻抗的 S 参数网络，可以在介质对象构造时指定端口阻抗 `z0_override`，或者可以手动覆盖阻抗。

In [ ]:
# override method
wr10_100_measured1 = wr10_100.copy()
wr10_100_measured1.z0[:,:] = 50
wr10_100_measured1.name = f'{wr10_100.name} override'
print(wr10_100_measured1)

# port impedance at media construction method
wr10_meas  =  RectangularWaveguide(f_wr10, a=to_meters(100, 'mil'), b=to_meters(50, 'mil'), ep_r=1.0,  rho=1.68e-08,
                                  z0_override = 50)
wr10_100_measured2 = wr10_meas.line(100e-3, unit = 'm', name = 'wr10 100mm z0_port')
print(wr10_100_measured2)

下图显示矩形波导测量的端口阻抗现在为 50 欧姆。插入损耗 S21 对于两种方法相等。然而，手动覆盖和构造时端口阻抗指定之间存在轻微的 S11 差异。

这是因为网络的默认 s 参数定义是 `s_def = 'power'`，且[特性阻抗具有虚部](https://scikit-rf.readthedocs.io/en/latest/examples/networktheory/Working%20with%20Complex%20Characteristic%20Impedances.html)。因此，完美匹配不是零而是复共轭。手动覆盖方法不会给出预期的结果。

In [ ]:
# prepare figure
fig4, axes = plt.subplots(2, 2, figsize = (8, 6))
gs = axes[1, 0].get_gridspec()
for ax in axes[1, :]:
    ax.remove()
axbig = fig4.add_subplot(gs[1, :])

# plot return loss
wr10_100_measured1.plot_s_mag(0, 0, ax=axes[0,0])
wr10_100_measured2.plot_s_mag(0, 0, ax=axes[0,0])

# plot insertion loss
wr10_100_measured1.plot_s_db(1, 0, ax=axes[0,1])
wr10_100_measured2.plot_s_db(1, 0, ax=axes[0,1])

# plot port and characteristic impedances
axbig.plot(wr10_100_measured1.frequency.f_scaled, wr10_100_measured1.z0[:, 0].real,
           marker = 'd', markevery = 30, label = f'line {wr10_100_measured1.name} z0')
axbig.plot(wr10_100_measured2.frequency.f_scaled, wr10_100_measured2.z0[:, 0].real,
           marker = 'x', markevery = 30, label = f'line {wr10_100_measured2.name} z0')
axbig.plot(wr10.frequency.f_scaled, wr10.z0.real, label = 'media wr10 z0')
axbig.set_ylabel('Impedance (Ohm)')
axbig.set_xlabel(f'Frequency ({mlin.frequency.unit})')
axbig.legend()

# resize plot nicely
fig4.tight_layout()

### 参数变化

可以改变介质的构造参数。例如，如果我们想知道基板相对介电常数的变化会如何影响微带线的*特性阻抗*和*传播常数*：

In [ ]:
#prepare
fig5, axes = plt.subplots(1, 2, figsize = (8, 3.5))

# plot
for ep_r in [4.0, 4.5, 5.0]:
    ml = MLine(f_mlin, w=3e-3, h=1.6e-3, t=35e-6, ep_r=ep_r,  rho=1.68e-08, z0_port=50)
    axes[0].plot(f_mlin.f_scaled, ml.z0.real, label=f'ep_r={ep_r:.1f}')
    axes[1].plot(f_mlin.f_scaled, ml.beta, label=f'ep_r={ep_r:.1f}')

axes[0].set_xlabel(f'Frequency ({f_mlin.unit})')
axes[0].set_ylabel('Characteristic Impedance (Ohm)')
axes[0].legend()
axes[1].set_xlabel(f'Frequency ({f_mlin.unit})')
axes[1].set_ylabel('Propagation Constant (rad/m)')

axes[1].legend()

# resize plot nicely
fig5.tight_layout()

更多详细示例说明如何创建各种类型的 [Media](../api/media/index.rst) 
对象如下。支持的介质列表在 [Media](../api/media/index.rst) API 页面中。如果需要执行复杂的电路设计，**skrf** 的网络创建和连接语法会很繁琐。**skrf** 的综合能力更适合脚本化应用，如校准、优化或批处理。

## 自由空间中的硅板

从 10 到 20 GHz 的自由空间中的平面波。

In [ ]:
from skrf.media import Freespace

freq = Frequency(10, 20, 101, 'GHz')
air =  Freespace(freq)
air

In [ ]:
air.z0[:2] # 377 Ohm baby!

In [ ]:
# plane wave in Si
si = Freespace(freq, ep_r = 11.2)
si.z0[:3] # ~110 Ohm


模拟半空间中的 1 厘米硅板，

In [ ]:
slab = air.thru() ** si.line(1, unit = 'cm') ** air.thru()
slab.plot_s_db(n = 0)

## 网络综合

网络通过 Media 对象的方法创建。要为微带线创建 1 端口短路，

In [ ]:
mlin_meas.short(name = 'short')

或者创建一段 $90^{\circ}$ 微带线，

In [ ]:
mlin_meas.line(d = 90, unit = 'deg', name = 'line')

## 构建电路


可以通过连接一系列网络来制作复杂电路。让我们构建一个 $90^{\circ}$ 微带线延迟短路。

In [ ]:
delay_short = mlin_meas.line(d = 90, unit = 'deg') ** mlin_meas.short()
delay_short.name = 'delay short'
delay_short

当需要将具有超过 2 个端口的 `Networks` 连接在一起时，使用
`rf.connect()`。要创建并联延迟开路的双端口网络，您可以创建一个理想的 3 路分路器（一个'三通'）并将延迟开路连接到其一个端口，
	

In [ ]:
tee = mlin_meas.tee()
delay_open = mlin_meas.delay_open(40, unit = 'deg')
shunt_open = rf.network.connect(tee, 1, delay_open, 0)

并联添加网络非常常见，有一个 `Media.shunt()` 函数可以执行此操作，

In [ ]:
mlin_meas.shunt(delay_open)

如果特定电路经常被创建，使用函数创建电路可能是有意义的。这可以使用 `lambda` 最快速地完成

In [ ]:
delay_short = lambda d: mlin_meas.line(d, unit = 'deg') ** mlin_meas.short()  # noqa: E731
delay_short(90)

更有用的示例可能是创建并联短截线调谐器的函数，
该函数适用于任何介质对象。

In [ ]:
def shunt_stub(med, d0, d1):
    return med.line(d0, unit = 'deg')**med.shunt_delay_open(d1, unit = 'deg')

shunt_stub(mlin_meas, 10, 90)

这种方法适用于设计优化。

## 设计优化


可以使用 `scipy` 的优化器能力来自动化网络设计。在此示例中，scikit-rf 用于自动化单短截线阻抗匹配网络设计。首先，我们创建一个'成本'函数，返回我们想要最小化的东西，例如频带中心的反射系数幅度。然后，使用 scipy 的最小化算法之一来确定短截线长度的最佳参数以最小化此成本。

In [ ]:
from scipy.optimize import fmin

from skrf.media import CPW

freq = Frequency(75, 110, 101, 'GHz')
cpw = CPW(freq, w = 10e-6, s = 5e-6, ep_r = 10.6, z0_port = 50)

# the load we are trying to match
load = cpw.load(.2+.2j)

# single stub circuit generator function
def shunt_stub(med, d0, d1):
    return med.line(d0, unit = 'deg') ** med.shunt_delay_open(d1, unit = 'deg')


# define the cost function we want to minimize (this uses sloppy namespace)
def cost(d):
    # prevent negative length lines, returning high cost
    if d[0] < 0 or d[1] < 0:
        return 1e3
    return (shunt_stub(cpw, d[0], d[1]) ** load)[100].s_mag.squeeze()

# initial guess of optimal delay lengths in degrees
d0 = 120,40 # initial guess

#determine the optimal delays
d_opt = fmin(cost, (120, 40))

d_opt